# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors — Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and visualize the [FAIR²](https://sen.science/doi/10.71728/senscience.qs2f-h81p) dataset using the `mlcroissant` library.

### Dataset Source
Schema: [https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)

This interactive notebook will use the Croissant schema as a guide to understand the dataset structure and process data for analysis.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading

We use `mlcroissant` to load metadata and data records from the Croissant schema. This approach ensures we access the dataset in a standards-based, re-usable manner.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# --- Dataset Croissant Schema URL ---
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print the dataset title and description
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Let's examine the available Record Sets, their `@id` values, and the fields they contain, all according to the Croissant schema. All entities will be referenced by their `@id`.

In [ ]:
# List available record sets and their field @ids
record_sets = list(dataset.record_sets)
print('Record sets:')
for rs in record_sets:
    print(f" - Name: {rs.name}")
    print(f"   @id: {rs['@id']}")
    print(f"   Fields:")
    for f in rs.fields:
        print(f"     - {f.name} (@id: {f['@id']}, dataType: {getattr(f, 'data_type', None)})")
    print()

## 3. Data Extraction

We will extract data from each available Record Set using their `@id` fields into Pandas DataFrames for further analysis.

In [ ]:
# Gather all record set @ids
record_set_ids = [rs['@id'] for rs in dataset.record_sets]

# Extract each record set into a pandas DataFrame keyed by @id
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# Display columns for the primary record set (choose first as example)
if record_set_ids:
    first_rs = record_set_ids[0]  # Use this as primary for demonstration
    print(f"Columns for record set {first_rs}:")
    print(dataframes[first_rs].columns.tolist())
    display(dataframes[first_rs].head())

## 4. Exploratory Data Analysis (EDA)

We will demonstrate filtering on a numeric column, normalizing its values, and grouping by a relevant attribute. Please use the `@id` of all fields for precise referencing.

In [ ]:
# Inspect columns in the main record set DataFrame
main_rs_id = record_set_ids[0]
df = dataframes[main_rs_id]
print(f"Columns in main record set ({main_rs_id}):")
print(df.columns.tolist())

# Find a likely numeric field by referencing the schema fields with data_type 'Integer' or 'Float'
# Will pick the first one found as an example
numeric_field_id = None
group_field_id = None

for rs in dataset.record_sets:
    if rs['@id'] == main_rs_id:
        for f in rs.fields:
            # Use only first integer/float field for demonstration
            if numeric_field_id is None and getattr(f, 'data_type', None) in ('Integer', 'Float', 'Number'):
                numeric_field_id = f['@id']
            if group_field_id is None and getattr(f, 'data_type', None) == 'Text':
                group_field_id = f['@id']

print(f"Using numeric field: {numeric_field_id}")
print(f"Using group field: {group_field_id}")

# Perform EDA only if numeric_field_id is available and present in columns
if numeric_field_id in df.columns:
    # Make sure column is numeric
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
        filtered_df[numeric_field_id].std()
    )
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by the group_field if it is present
    if group_field_id in df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
        display(grouped_df.head())
    else:
        print(f"Group field {group_field_id} not present in columns.")
else:
    print(f"Numeric field {numeric_field_id} not found in record set columns.")

## 5. Visualization

Let's visualize the distribution of the selected numeric field and its breakdown by grouping field, using the `@id` of each field for column selection.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id in df.columns and df[numeric_field_id].notnull().any():
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    if group_field_id in df.columns and df[group_field_id].nunique() < 10:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} grouped by {group_field_id}")
        plt.show()
else:
    print(f"No suitable numeric field {numeric_field_id} for visualization.")

## 6. Conclusion

This notebook demonstrates how to load, inspect, and analyze a Croissant-formatted biomedical dataset using `mlcroissant` and reference all fields/columns by their `@id`. Such a pipeline ensures that dataset processing is reproducible, standard-compliant, and keeps provenance with the data schema. For advanced analyses, further steps like feature engineering, modeling, or hypothesis testing can be added while referencing fields and sets by their `@id` as shown. 